
# Exercícios de Pandas com Base de Dados do Spotify

In [ ]:
import numpy as np
import pandas as pd
import kagglehub
import ssl

# Desabilitar verificação SSL (necessário para redes corporativas)
ssl._create_default_https_context = ssl._create_unverified_context

# Baixando a última versão no kaggle
import os
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
os.environ['REQUESTS_CA_BUNDLE'] = ''

path = kagglehub.dataset_download("maharshipandya/-spotify-tracks-dataset", force_download=True)

#leitura do arquivo

dados_completos = []
with open(path+"/dataset.csv", "r", encoding="utf-8") as arquivo:
    linhas = arquivo.readlines() # lê todas as linhas
    linhas = linhas[1:] # remove cabeçalho
    for linha in linhas:
        try:
            linha = linha.strip() # remove quebra de linha
            colunas = linha.split(",") # separa colunas
            # colunas numéricas ( popularity, duration_ms, danceability, energy, key, loudness, mode, speechiness, acousticness, instrumentalness liveness, valence, tempo, time_signature)
            valores = [
                colunas[2],   # track_name
                colunas[1],   # artist
                colunas[20],  # genre

                float(colunas[5]), float(colunas[6]), float(colunas[8]), float(colunas[9]), float(colunas[10]), float(colunas[11]), float(colunas[12]), float(colunas[13]), float(colunas[14]), float(colunas[15]), float(colunas[16]), float(colunas[17]), float(colunas[18]), float(colunas[19])
            ]
            dados_completos.append(valores)
        except:
            pass

dados = np.array(dados_completos, dtype=object) # converte para ndarray

# nomes das colunas
colunas_df = [

    "track_name",
    "artist",
    "genre",

    "popularity",
    "duration_ms",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature"

]

# converte ndarray para DataFrame
df = pd.DataFrame(dados, columns=colunas_df)

print(df.head())

c:\Users\leonardo.flores\Desktop\UFOP\PROGRAMAÇÃO PARA CIÊNCIA DE DADOS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SSLError: HTTPSConnectionPool(host='api.kaggle.com', port=443): Max retries exceeded with url: /v1/datasets.DatasetApiService/GetDataset (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)')))


1) A plataforma Spotify deseja identificar quais músicas possuem maior destaque entre os usuários. Utilize recursos do pandas: seleção de colunas, filtragem booleana, `sort_values()` e `head()`. Utilizando a base de dados fornecida selecione apenas as colunas: `track_name`, `artist`, `genre` e `popularity`. Filtre apenas as músicas com popularidade maior que 80 e apresente os resultados ordenados da maior para a menor popularidade.




## Gabarito

In [ ]:
df['popularity'] = pd.to_numeric(df['popularity'])

resultado = df[['track_name', 'artist', 'genre', 'popularity']]
resultado = resultado[resultado['popularity'] > 80]
resultado = resultado.sort_values('popularity', ascending=False)

print(resultado.head())


2) O Spotify deseja criar categorias para classificar automaticamente o perfil musical das músicas com base nos atributos: `danceability`, `energy` e `valence`. Crie uma nova coluna chamada `perfil_musical`, que com base na média destes atributos, classifique cada música como: `"Calma"` (<0.4), `"Moderada"` (<0.7) e `"Agitada"` (>0.7). Utilize recursos do pandas: `apply()`, criação de funções e criação de novas colunas.


## Gabarito

In [ ]:
for col in ['danceability', 'energy', 'valence']:
    df[col] = pd.to_numeric(df[col])

def classificar_perfil(row):
    media = (row['danceability'] + row['energy'] + row['valence']) / 3
    if media < 0.4:
        return 'Calma'
    elif media < 0.7:
        return 'Moderada'
    else:
        return 'Agitada'

df['perfil_musical'] = df.apply(classificar_perfil, axis=1)

print(df[['track_name', 'artist', 'danceability', 'energy', 'valence', 'perfil_musical']].head(10))

3) O Spotify deseja analisar a evolução dos lançamentos musicais ao longo do tempo. Como a base de dados não possui uma coluna de datas reais de lançamento, crie uma coluna fictícia chamada `release_date` utilizando datas sequenciais. Utilize recursos do pandas: `pd.date_range()`, `pd.to_datetime()`, `indexação temporal`, `slicing temporal`, `loc[]` e `sort_index()`. Em seguida:
- converta a coluna release_date para o formato de data;
- defina essa coluna como índice temporal;
- selecione apenas músicas lançadas em 2020;
- filtre músicas lançadas entre 2018 e 2021;
- apresente os resultados em ordem cronológica.


## Gabarito

In [ ]:
df['release_date'] = pd.date_range(start='2010-01-01', periods=len(df), freq='D')

df['release_date'] = pd.to_datetime(df['release_date'])

df = df.set_index('release_date')

df = df.sort_index()

lancamentos_2020 = df.loc['2020']
print("Músicas lançadas em 2020:")
print(lancamentos_2020[['track_name', 'artist', 'genre']].head())

lancamentos_2018_2021 = df.loc['2018':'2021']
print("\nMúsicas lançadas entre 2018 e 2021:")
print(lancamentos_2018_2021[['track_name', 'artist', 'genre']].head())